In [27]:
import h5py
import numpy as np

In [28]:
f = h5py.File("pcas_ocbs_simulated_data.hdf5", "r")
t  = f['target']
sd = t['simulated_data']

# ── file structure overview ───────────────────────────────────────────────────
print("=== target attributes ===")
for k, v in t.attrs.items():
    print(f"  {k}: {v}")

print("\n=== target datasets ===")
for name in t.keys():
    if name == 'simulated_data':
        continue
    d = t[name]
    print(f"  {name}: shape={d.shape}  dtype={d.dtype}")

print("\n=== simulated_data datasets ===")
for name in sd.keys():
    d = sd[name]
    print(f"  {name}: shape={d.shape}  dtype={d.dtype}")

=== target attributes ===
  light_source: Thorlabs SLD830S-A10 @ I=140mA, T=25C
  num_orientations: 10
  polarization_state: left-handed circular: E0_theta=1/sqrt(2), E0_phi=1j/sqrt(2)

=== target datasets ===
  ab_ratio_list: shape=(1,)  dtype=float64
  bc_ratio_list: shape=(1,)  dtype=float64
  gre_beta_list: shape=(1,)  dtype=float64
  m_p_xyz_list: shape=(3, 3)  dtype=complex128
  r_v_base_list: shape=(2,)  dtype=float64
  wl_m_m_pairs: shape=(3, 2)  dtype=float64

=== simulated_data datasets ===
  C_abs: shape=(3, 3, 2, 1, 1, 1, 10)  dtype=float64
  C_abs_mie: shape=(3, 3, 2, 1, 1, 1)  dtype=float64
  C_ext: shape=(3, 3, 2, 1, 1, 1, 10)  dtype=float64
  C_ext_mie: shape=(3, 3, 2, 1, 1, 1)  dtype=float64
  Euler_angles: shape=(3, 3, 2, 1, 1, 1, 10, 3)  dtype=float64
  S_bk_OCBS: shape=(3, 3, 2, 1, 1, 1, 10)  dtype=complex128
  S_bk_OCBS_mie: shape=(3, 3, 2, 1, 1, 1)  dtype=complex128
  S_fw_PCAS_mie: shape=(3, 3, 2, 1, 1, 1)  dtype=complex128
  S_fw_PCAS_phi: shape=(3, 3, 2, 1, 1, 

In [29]:
# ── sweep parameter summary ───────────────────────────────────────────────────
wl_m_m_pairs  = t['wl_m_m_pairs'][:]    # (N_pairs, 2)
m_p_xyz_list  = t['m_p_xyz_list'][:]    # (N_m_p,   3)
r_v_base_list = t['r_v_base_list'][:]
bc_ratio_list = t['bc_ratio_list'][:]
ab_ratio_list = t['ab_ratio_list'][:]
gre_beta_list = t['gre_beta_list'][:]
num_ori       = int(t.attrs['num_orientations'])

print("=== sweep parameters ===")
print(f"  wl_m_m_pairs  ({len(wl_m_m_pairs)} pairs):")
for i, (wl, mm) in enumerate(wl_m_m_pairs):
    print(f"    [{i}]  wl_0={wl:.4f} um   m_m={mm:.4f}")

print(f"  m_p_xyz_list  ({len(m_p_xyz_list)} entries):")
for i, mp in enumerate(m_p_xyz_list):
    print(f"    [{i}]  m_p_xyz={mp}")

print(f"  r_v_base_list : {r_v_base_list}")
print(f"  bc_ratio_list : {bc_ratio_list}")
print(f"  ab_ratio_list : {ab_ratio_list}")
print(f"  gre_beta_list : {gre_beta_list}")
print(f"  num_orientations : {num_ori}")

=== sweep parameters ===
  wl_m_m_pairs  (3 pairs):
    [0]  wl_0=0.5320 um   m_m=1.3310
    [1]  wl_0=0.6330 um   m_m=1.3300
    [2]  wl_0=0.8337 um   m_m=1.3290
  m_p_xyz_list  (3 entries):
    [0]  m_p_xyz=[1.5+0.j 1.5+0.j 1.5+0.j]
    [1]  m_p_xyz=[1.5+0.05j 1.5+0.05j 1.5+0.05j]
    [2]  m_p_xyz=[1.6+0.j 1.5+0.j 1.4+0.j]
  r_v_base_list : [0.15 0.2 ]
  bc_ratio_list : [1.]
  ab_ratio_list : [1.]
  gre_beta_list : [0.]
  num_orientations : 10


In [30]:
# ── completion status ─────────────────────────────────────────────────────────
# S_fw_PCAS_mie.imag != 0 when that condition has been computed.
mie_s = sd['S_fw_PCAS_mie'][:]    # (N_pairs, N_m_p, N_rv, N_bc, N_ab, N_bt)
done  = (mie_s.imag != 0.0)       # bool array, same shape

N_pairs, N_mp = mie_s.shape[:2]
N_total = done.size
N_done  = done.sum()
print(f"Completed: {N_done} / {N_total} conditions\n")

for i_pair, (wl, mm) in enumerate(wl_m_m_pairs):
    for i_mp, mp in enumerate(m_p_xyz_list):
        frac = done[i_pair, i_mp].sum()
        total = done[i_pair, i_mp].size
        mark = "✓" if frac == total else ("…" if frac > 0 else "✗")
        print(f"  {mark} pair[{i_pair}] wl={wl:.4f} m_m={mm:.4f}  "
              f"m_p[{i_mp}] = {mp}  →  {frac}/{total}")

Completed: 18 / 18 conditions

  ✓ pair[0] wl=0.5320 m_m=1.3310  m_p[0] = [1.5+0.j 1.5+0.j 1.5+0.j]  →  2/2
  ✓ pair[0] wl=0.5320 m_m=1.3310  m_p[1] = [1.5+0.05j 1.5+0.05j 1.5+0.05j]  →  2/2
  ✓ pair[0] wl=0.5320 m_m=1.3310  m_p[2] = [1.6+0.j 1.5+0.j 1.4+0.j]  →  2/2
  ✓ pair[1] wl=0.6330 m_m=1.3300  m_p[0] = [1.5+0.j 1.5+0.j 1.5+0.j]  →  2/2
  ✓ pair[1] wl=0.6330 m_m=1.3300  m_p[1] = [1.5+0.05j 1.5+0.05j 1.5+0.05j]  →  2/2
  ✓ pair[1] wl=0.6330 m_m=1.3300  m_p[2] = [1.6+0.j 1.5+0.j 1.4+0.j]  →  2/2
  ✓ pair[2] wl=0.8337 m_m=1.3290  m_p[0] = [1.5+0.j 1.5+0.j 1.5+0.j]  →  2/2
  ✓ pair[2] wl=0.8337 m_m=1.3290  m_p[1] = [1.5+0.05j 1.5+0.05j 1.5+0.05j]  →  2/2
  ✓ pair[2] wl=0.8337 m_m=1.3290  m_p[2] = [1.6+0.j 1.5+0.j 1.4+0.j]  →  2/2


In [31]:
# ── select condition to inspect ───────────────────────────────────────────────
# Change these indices to browse different conditions.
i_pair = 0   # index into wl_m_m_pairs
i_mp   = 1   # index into m_p_xyz_list
i_rv   = 1   # index into r_v_base_list
i_bc   = 0   # index into bc_ratio_list
i_ab   = 0   # index into ab_ratio_list
i_bt   = 0   # index into gre_beta_list

idx6 = (i_pair, i_mp, i_rv, i_bc, i_ab, i_bt)

wl_0, m_m = wl_m_m_pairs[i_pair]
m_p_xyz   = m_p_xyz_list[i_mp]
r_v_base  = r_v_base_list[i_rv]
bc_ratio  = bc_ratio_list[i_bc]
ab_ratio  = ab_ratio_list[i_ab]
gre_beta  = gre_beta_list[i_bt]

print(f"Selected condition  idx6={idx6}")
print(f"  wl_0={wl_0:.4f} um   m_m={m_m:.4f}")
print(f"  m_p_xyz = {m_p_xyz}")
print(f"  r_v_base={r_v_base:.3f}  bc={bc_ratio:.1f}  ab={ab_ratio:.1f}  beta={gre_beta:.2f}")
print(f"  r_ve = {sd['r_ve'][idx6[2:]]:.4f} um")
print()

Selected condition  idx6=(0, 1, 1, 0, 0, 0)
  wl_0=0.5320 um   m_m=1.3310
  m_p_xyz = [1.5+0.05j 1.5+0.05j 1.5+0.05j]
  r_v_base=0.200  bc=1.0  ab=1.0  beta=0.00
  r_ve = 0.2027 um



In [32]:
# ── per-orientation DDA results for selected condition ────────────────────────
euler    = sd['Euler_angles'   ][idx6]          # (N_ori, 3)
C_abs    = sd['C_abs'          ][idx6]          # (N_ori,)
C_ext    = sd['C_ext'          ][idx6]          # (N_ori,)
S_fw_th  = sd['S_fw_PCAS_theta'][idx6]          # (N_ori,) complex
S_fw_ph  = sd['S_fw_PCAS_phi'  ][idx6]          # (N_ori,) complex
S_bk     = sd['S_bk_OCBS'      ][idx6]          # (N_ori,) complex

print("=== DDA per-orientation results ===")
print(f"  {'quantity':<24}  {'mean':>14}  {'std':>14}")
print(f"  {'-'*54}")

def _stat_real(label, arr):
    valid = arr[~np.isnan(arr.real)].real
    if len(valid):
        print(f"  {label:<24}  {np.mean(valid):>14.4e}  {np.std(valid):>14.4e}")
    else:
        print(f"  {label:<24}  {'(no data)':>14}")

def _stat_ri(label, arr):
    valid = arr[~np.isnan(arr.real)]
    if len(valid):
        for part, vals in [("Re", valid.real), ("Im", valid.imag)]:
            lbl = f"{part}({label})"
            print(f"  {lbl:<24}  {np.mean(vals):>14.4e}  {np.std(vals):>14.4e}")
    else:
        for part in ("Re", "Im"):
            print(f"  {part+'('+label+')':<24}  {'(no data)':>14}")

_stat_real("C_abs [um^2]",    C_abs)
_stat_real("C_ext [um^2]",    C_ext)
_stat_ri(  "S_fw_theta [um]", S_fw_th)
_stat_ri(  "S_fw_phi [um]",   S_fw_ph)
_stat_ri(  "S_bk_OCBS [um]",  S_bk)

print(f"\nEuler angles (first 5 rows) [rad]:\n{euler[:5]}")
#全配向をまとめて比較したい場合は、euler[:5] の 5 の数を増やすか、euler[:] で全行を表示する。

=== DDA per-orientation results ===
  quantity                            mean             std
  ------------------------------------------------------
  C_abs [um^2]                  4.1226e-02      2.0818e-05
  C_ext [um^2]                  7.5432e-02      1.9596e-05
  Re(S_fw_theta [um])           1.5113e-01      2.6816e-04
  Im(S_fw_theta [um])           9.4296e-02      1.8716e-04
  Re(S_fw_phi [um])             1.5113e-01      2.3932e-04
  Im(S_fw_phi [um])             9.4424e-02      1.6072e-04
  Re(S_bk_OCBS [um])           -2.4509e-03      1.2593e-03
  Im(S_bk_OCBS [um])           -9.1974e-03      1.1083e-03

Euler angles (first 5 rows) [rad]:
[[2.97491496 2.09901332 2.26775347]
 [0.2422558  2.67786387 5.28958156]
 [4.19922952 0.35260884 4.91116139]
 [4.26973942 1.98879978 1.13154191]
 [0.29857753 0.68282454 0.50278482]]


In [33]:
# ── individual orientation inspection ────────────────────────────────────────
# Set i_ori to the orientation index you want to inspect (0 .. num_ori-1).
# Requires Cell 5 to have been run first (euler, C_abs, C_ext, S_fw_th, S_fw_ph, S_bk).
i_ori = 0

alpha_deg, beta_deg, gamma_deg = np.degrees(euler[i_ori])
print(f"=== orientation i_ori={i_ori} ===")
print(f"  Euler angles (alpha, beta, gamma) = "
      f"({alpha_deg:.1f}°, {beta_deg:.1f}°, {gamma_deg:.1f}°)")
print()
print(f"  C_abs      = {C_abs[i_ori]:.4e}  um^2")
print(f"  C_ext      = {C_ext[i_ori]:.4e}  um^2")
print(f"  S_fw_theta = {S_fw_th[i_ori]:.4g}  um")
print(f"  S_fw_phi   = {S_fw_ph[i_ori]:.4g}  um")
print(f"  S_bk_OCBS  = {S_bk[i_ori]:.4g}  um")


=== orientation i_ori=0 ===
  Euler angles (alpha, beta, gamma) = (170.5°, 120.3°, 129.9°)

  C_abs      = 4.1265e-02  um^2
  C_ext      = 7.5435e-02  um^2
  S_fw_theta = 0.1513+0.09448j  um
  S_fw_phi   = 0.1512+0.09425j  um
  S_bk_OCBS  = -0.0008516-0.007842j  um


In [34]:
# ── Mie reference values for selected condition ───────────────────────────────
C_abs_mie   = sd['C_abs_mie'    ][idx6]
C_ext_mie   = sd['C_ext_mie'    ][idx6]
S_fw_mie    = sd['S_fw_PCAS_mie'][idx6]
S_bk_mie    = sd['S_bk_OCBS_mie'][idx6]

print("=== Mie reference (volume-equivalent sphere) ===")
print(f"  C_abs_mie   = {C_abs_mie:.4e} um^2")
print(f"  C_ext_mie   = {C_ext_mie:.4e} um^2")
print(f"  S_fw_mie    = {S_fw_mie:.4g}")
print(f"  S_bk_mie    = {S_bk_mie:.4g}")

=== Mie reference (volume-equivalent sphere) ===
  C_abs_mie   = 4.1399e-02 um^2
  C_ext_mie   = 7.5675e-02 um^2
  S_fw_mie    = 0.1513+0.09467j
  S_bk_mie    = -0.003351-0.009692j


In [35]:
# ── DDA vs Mie comparison for selected condition ─────────────────────────────
# Orientation-averaged DDA values vs Mie reference.
# Only meaningful after DDA computation is complete for this condition.
print("=== DDA (orientation mean) vs Mie ===")
print(f"  {'quantity':<24}  {'DDA mean':>14}  {'Mie':>14}  {'|DDA-Mie|/|Mie|':>16}")
print(f"  {'-'*72}")

def _cmp_real(label, dda_arr, mie_val):
    """Real-valued quantity (C_abs, C_ext)."""
    valid = dda_arr[~np.isnan(dda_arr.real)]
    if len(valid) == 0 or mie_val == 0:
        print(f"  {label:<24}  {'(no data)':>14}")
        return
    dda_mean = float(np.mean(valid).real)
    rel_err  = abs(dda_mean - mie_val) / abs(mie_val)
    print(f"  {label:<24}  {dda_mean:>14.4e}  {float(mie_val):>14.4e}  {rel_err:>16.4f}")

def _cmp_ri(label, dda_arr, mie_val):
    """Complex quantity: compare Re and Im parts separately."""
    valid = dda_arr[~np.isnan(dda_arr.real)]
    if len(valid) == 0:
        for part in ("Re", "Im"):
            print(f"  {part+'('+label+')':<24}  {'(no data)':>14}")
        return
    dda_mean = np.mean(valid)
    for part, dda_part, mie_part in [("Re", dda_mean.real, mie_val.real),
                                      ("Im", dda_mean.imag, mie_val.imag)]:
        lbl = f"{part}({label})"
        if mie_part == 0:
            print(f"  {lbl:<24}  {dda_part:>14.4e}  {mie_part:>14.4e}  {'(Mie=0)':>16}")
        else:
            rel_err = abs(dda_part - mie_part) / abs(mie_part)
            print(f"  {lbl:<24}  {dda_part:>14.4e}  {mie_part:>14.4e}  {rel_err:>16.4f}")

_cmp_real("C_abs [um^2]",   C_abs,   C_abs_mie)
_cmp_real("C_ext [um^2]",   C_ext,   C_ext_mie)
_cmp_ri(  "S_fw_theta [um]", S_fw_th, S_fw_mie)
_cmp_ri(  "S_fw_phi [um]",   S_fw_ph, S_fw_mie)
_cmp_ri(  "S_bk [um]",       S_bk,    S_bk_mie)

=== DDA (orientation mean) vs Mie ===
  quantity                        DDA mean             Mie   |DDA-Mie|/|Mie|
  ------------------------------------------------------------------------
  C_abs [um^2]                  4.1226e-02      4.1399e-02            0.0042
  C_ext [um^2]                  7.5432e-02      7.5675e-02            0.0032
  Re(S_fw_theta [um])           1.5113e-01      1.5131e-01            0.0012
  Im(S_fw_theta [um])           9.4296e-02      9.4665e-02            0.0039
  Re(S_fw_phi [um])             1.5113e-01      1.5131e-01            0.0012
  Im(S_fw_phi [um])             9.4424e-02      9.4665e-02            0.0025
  Re(S_bk [um])                -2.4509e-03     -3.3511e-03            0.2686
  Im(S_bk [um])                -9.1974e-03     -9.6920e-03            0.0510


In [36]:
f.close()